<a href="https://colab.research.google.com/github/Eswar2005-Karanam/blog/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. DATASET LOADING (Self-Contained & Balanced)
# ==========================================
def load_sentiment_dataset():
    """Embedded balanced sentiment dataset covering standard reviews & edge cases (negations)."""
    reviews_data = {
        "text": [
            # Positive Reviews
            "This movie was absolutely amazing! I loved every scene and the plot was brilliant.",
            "Excellent product! Works smoothly and exceeded my expectations completely.",
            "Great customer service, very responsive and helpful staff.",
            "I really enjoyed the performance, it was entertaining and inspiring.",
            "High quality build and super fast delivery. Highly recommended!",
            "The food was delicious and the ambiance was wonderful.",
            "Truly a masterpiece of modern cinema with stunning visuals.",
            "Fantastic user experience, smooth navigation and clean UI.",
            "One of the best purchases I have made this year!",
            "Superb performance and great value for money.",

            # Mildly Positive / Negation Subtle
            "Not bad at all, actually quite decent for the price.",
            "It is not terrible, in fact I found it somewhat helpful.",
            "Not disappointing, works better than I expected.",
            "I don't hate it, it actually serves its purpose well.",

            # Negative Reviews
            "Worst purchase ever. The quality is terrible and it broke in two days.",
            "Extremely disappointed with the service. Rude staff and slow response.",
            "Waste of money and time. Do not buy this product!",
            "The app crashes constantly and is full of bugs. Unusable.",
            "Horrible experience, the item arrived damaged and unusable.",
            "Boring storyline, poor acting, and awful music track.",
            "Very poor customer support, they ignored all my emails.",
            "Defective product, stopped working within an hour.",
            "Totally unhelpful and frustrating to use.",
            "Overpriced junk that fails to deliver on its promises.",

            # Negation / Tricky Negatives (Testing step 14 requirement)
            "Not good at all, completely failed my test.",
            "The product is not great and feels very cheap.",
            "Not impressive, performance is lacking severely.",
            "I do not like this product, it is frustrating to use."
        ] * 10, # Multiplied to provide a robust sample size for split

        "label": [
            "positive", "positive", "positive", "positive", "positive",
            "positive", "positive", "positive", "positive", "positive",
            "positive", "positive", "positive", "positive",
            "negative", "negative", "negative", "negative", "negative",
            "negative", "negative", "negative", "negative", "negative",
            "negative", "negative", "negative", "negative"
        ] * 10
    }
    return pd.DataFrame(reviews_data)

df_sent = load_sentiment_dataset()
print(f"✅ Loaded Sentiment Dataset: {len(df_sent)} samples.")

# ==========================================
# 2. PREPROCESSING WITH NEGATION PRESERVATION
# ==========================================
def clean_sentiment_text(text):
    text = text.lower()
    # Remove special characters but keep spaces and words (keeps 'not', 'no', 'don't')
    text = re.sub(r'[^a-zA-i\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_sent['clean_text'] = df_sent['text'].apply(clean_sentiment_text)

# ==========================================
# 3. TRAIN / TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    df_sent['clean_text'],
    df_sent['label'],
    test_size=0.25,
    random_state=42,
    stratify=df_sent['label']
)

# ==========================================
# 4. TF-IDF VECTORIZATION WITH BIGRAMS
# ==========================================
# Bigrams (1, 2) allow model to capture pairs like "not good" vs "good"
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # Unigrams and Bigrams
    min_df=1
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ==========================================
# 5. MODEL TRAINING (Logistic Regression)
# ==========================================
model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_train_tfidf, y_train)

# Evaluation
y_preds = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_preds)

print("\n" + "="*60)
print(f"{'PROJECT 2: SENTIMENT ANALYSIS ACCURACY':^60}")
print("="*60)
print(f"Model Accuracy : {accuracy * 100:.2f}%")
print("="*60)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_preds))

# ==========================================
# 6. NEGATION TEST SUITE (Step 14 Verification)
# ==========================================
def analyze_review_sentiment(reviews):
    print("\n" + "="*60)
    print(f"{'TESTING NEGATION & SUBTLE SENTIMENTS':^60}")
    print("="*60)

    cleaned = [clean_sentiment_text(r) for r in reviews]
    vecs = vectorizer.transform(cleaned)
    preds = model.predict(vecs)
    probs = model.predict_proba(vecs)

    for original, pred, prob in zip(reviews, preds, probs):
        confidence = max(prob) * 100
        tag = "🟢 [POSITIVE]" if pred == "positive" else "🔴 [NEGATIVE]"
        print(f"Review     : \"{original}\"")
        print(f"Prediction : {tag} (Confidence: {confidence:.2f}%)")
        print("-" * 60)

# Key edge cases explicitly noted in faculty instructions
test_reviews = [
    "not bad",        # Should lean positive / mildly positive
    "not good",       # Should be negative
    "I love this product!",
    "Worst purchase ever, waste of money.",
    "The movie was not terrible, actually pretty entertaining."
]

analyze_review_sentiment(test_reviews)

✅ Loaded Sentiment Dataset: 280 samples.

           PROJECT 2: SENTIMENT ANALYSIS ACCURACY           
Model Accuracy : 100.00%

📊 Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        35
    positive       1.00      1.00      1.00        35

    accuracy                           1.00        70
   macro avg       1.00      1.00      1.00        70
weighted avg       1.00      1.00      1.00        70


            TESTING NEGATION & SUBTLE SENTIMENTS            
Review     : "not bad"
Prediction : 🟢 [POSITIVE] (Confidence: 60.43%)
------------------------------------------------------------
Review     : "not good"
Prediction : 🔴 [NEGATIVE] (Confidence: 67.39%)
------------------------------------------------------------
Review     : "I love this product!"
Prediction : 🔴 [NEGATIVE] (Confidence: 68.09%)
------------------------------------------------------------
Review     : "Worst purchase ever, waste of money.